# PATCHR Server on Google Colab

Run the PATCHR inpainting server on Colab's free GPU and get a public URL to connect from **PATCHR-Studio**.

**Steps:**
1. Go to **Runtime > Change runtime type > GPU** (T4)
2. Run all cells below in order
3. Copy the public URL into PATCHR-Studio

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Install PATCHR

In [ ]:
!git clone https://github.com/DeepFoldProtein/patchr.git
%cd patchr
!pip install -e .[cuda] -q

## 3. Start server and tunnel

This starts the PATCHR server and creates a public URL using [Cloudflare Tunnel](https://developers.cloudflare.com/cloudflare-one/connections/connect-networks/) (no account or token required).

In [ ]:
import subprocess
import time
import re

# Install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Start the PATCHR server
server_process = subprocess.Popen(
    ["python", "-m", "boltz.server", "--port", "31212", "--device-id", "0"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print("Starting PATCHR server...")
time.sleep(10)

# Start cloudflared tunnel
tunnel_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:31212"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(5)

# Read tunnel URL from stderr (cloudflared logs to stderr)
import threading

tunnel_url = None

def read_tunnel_output():
    global tunnel_url
    for line in tunnel_process.stderr:
        line = line.decode("utf-8", errors="ignore")
        match = re.search(r"(https://[\w-]+\.trycloudflare\.com)", line)
        if match:
            tunnel_url = match.group(1)
            break

t = threading.Thread(target=read_tunnel_output, daemon=True)
t.start()
t.join(timeout=15)

if tunnel_url:
    print("\n" + "=" * 60)
    print(f"  PATCHR Server is running!")
    print(f"  Public URL: {tunnel_url}")
    print(f"\n  Copy this URL into PATCHR-Studio")
    print("=" * 60 + "\n")
else:
    print("Could not retrieve tunnel URL. Check the logs below:")
    !cloudflared tunnel --url http://localhost:31212 2>&1 | head -20

## 4. Health check (optional)

Verify the server is running correctly.

In [ ]:
import requests

try:
    r = requests.get("http://localhost:31212/api/v1/health")
    print("Health check:", r.json())
except Exception as e:
    print(f"Server not ready yet: {e}")
    print("Wait a moment and try again.")

## 5. Stop server

Run this cell when you are done.

In [ ]:
tunnel_process.terminate()
server_process.terminate()
print("Server stopped.")